# Streaming Algorithms & Sketches

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/streaming-ml/02-streaming-algorithms

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

**The problem.** You cannot store a stream, so you store a small **sketch** that answers one query approximately. **The core idea:** pick a summary whose size is *logarithmic* in the stream, update it in O(1) per element, and accept a bounded error. We build three from scratch — a uniform **sample** (reservoir), a membership **filter** (Bloom), and a **distinct-count** estimator (Flajolet-Martin) — and check each against its theoretical guarantee. These power the features that feed streaming models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)
import hashlib

## 1. From scratch — reservoir sampling (Algorithm R)

Keep a uniform sample of `k` items in one pass: put the first `k` in the reservoir, then accept the `t`-th element with probability `k/t`, evicting a random slot. The claim to verify: after the pass, every element is in the reservoir with probability `k/n`.

In [ ]:
def reservoir_sample(stream, k, rng):
    res = []
    for t, x in enumerate(stream, start=1):
        if t <= k:
            res.append(x)
        elif rng.random() < k / t:
            res[rng.integers(k)] = x
    return res

## 2. The library way — validate uniformity empirically

There is no single stdlib streaming reservoir, but the *guarantee* is checkable: over many runs each of `n` items should appear a fraction `k/n` of the time. We assert the empirical inclusion rate matches `k/n`.

In [ ]:
rng = np.random.default_rng(0)
n, k, trials = 40, 5, 40000
counts = np.zeros(n)
for _ in range(trials):
    for x in reservoir_sample(range(n), k, rng):
        counts[x] += 1
emp = counts / trials
expected = k / n
print(f'expected inclusion k/n = {expected:.3f}')
print(f'empirical min/mean/max = {emp.min():.3f} / {emp.mean():.3f} / {emp.max():.3f}')
assert abs(emp.mean() - expected) < 0.01, 'mean inclusion must equal k/n'
assert emp.max() - emp.min() < 0.03, 'every element must be near-equally likely (uniform)'
print('reservoir sample is uniform (inclusion ≈ k/n for every element) ✓')

## 3. Bloom filter — membership with no false negatives

An `m`-bit array + `k` hashes. Insert sets `k` bits; query returns 'present' only if all `k` are set. False negatives are impossible; the false-positive rate is $p \approx (1 - e^{-kn/m})^k$, minimised at $k = (m/n)\ln 2$.

In [ ]:
class BloomFilter:
    def __init__(self, m, k):
        self.m, self.k = m, k
        self.bits = np.zeros(m, dtype=bool)
    def _idx(self, x):
        h = int(hashlib.md5(str(x).encode()).hexdigest(), 16)
        h1, h2 = h & 0xFFFFFFFF, (h >> 32) | 1   # double hashing
        for i in range(self.k):
            yield (h1 + i * h2) % self.m
    def add(self, x):
        for i in self._idx(x):
            self.bits[i] = True
    def __contains__(self, x):
        return all(self.bits[i] for i in self._idx(x))

### Check: predicted vs empirical false-positive rate

With 10 bits/element and the optimal `k`, the formula predicts ~0.8%. We insert `n` members (assert none are false negatives) and measure the false-positive rate on unseen keys.

In [ ]:
n, m = 2000, 20000                       # 10 bits per element
k = max(1, round((m / n) * np.log(2)))   # optimal k ~ 7
bf = BloomFilter(m, k)
for i in range(n):
    bf.add(('member', i))
assert all(('member', i) in bf for i in range(n)), 'Bloom filters never have false negatives'
fp = np.mean([('other', j) in bf for j in range(40000)])
predicted = (1 - np.exp(-k * n / m)) ** k
print(f'k={k}  predicted FP = {predicted:.4f}  empirical FP = {fp:.4f}')
assert abs(fp - predicted) < 0.01, 'empirical FP rate must match the formula'
print('Bloom false-positive rate matches (1 - e^(-kn/m))^k ✓')

## 4. Flajolet-Martin — counting distinct elements

Hash each element; track `R`, the max number of **trailing zeros** seen. Since a random hash ends in `>= r` zeros with probability `2^-r`, `d` distinct values give `R ~ log2(d)`, so the estimate is `2^R`. Duplicates never change `R`. One estimator is noisy, so we average many (the LogLog idea).

In [ ]:
def trailing_zeros(x):
    return (x & -x).bit_length() - 1 if x else 64

def fm_estimate(items, salt):
    R = 0
    for x in items:
        h = int(hashlib.sha1((str(x) + '|' + str(salt)).encode()).hexdigest(), 16)
        R = max(R, trailing_zeros(h))
    return 2 ** R

true_distinct = 5000
stream = list(np.random.randint(0, true_distinct, size=80000))  # heavy duplication
ests = [fm_estimate(stream, salt) for salt in range(96)]        # average many estimators
median_est = int(np.median(ests))
print(f'true distinct = {true_distinct}  |  median FM estimate = {median_est}')
assert 0.5 * true_distinct < median_est < 2 * true_distinct, 'FM should be within a factor of 2'
print('Flajolet-Martin estimates distinct count in O(log n) space ✓')

## 5. Visualize it — memory vs accuracy

The whole point of sketches is a favourable error/space trade. For the Bloom filter, more bits-per-element drives the false-positive rate down geometrically.

In [ ]:
ratios = np.array([4, 6, 8, 10, 12, 16])
def predicted_fp(bits_per):
    kk = max(1, round(bits_per * np.log(2)))
    return (1 - np.exp(-kk / bits_per)) ** kk
fps = [predicted_fp(r) for r in ratios]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.semilogy(ratios, fps, 'o-', color='#14b8a6')
ax.set_xlabel('bits per element (m/n)'); ax.set_ylabel('false-positive rate (log)')
ax.set_title('Bloom filter: error falls geometrically with memory', color='white')
ax.grid(alpha=0.2, which='both'); plt.show()

**What to notice:** each extra ~2 bits per element roughly halves the false-positive rate. Ten bits buys sub-1% error — a few bits to remember membership, versus storing the items themselves. Every sketch here trades a little accuracy for an exponential memory saving.

## 6. Gotchas & tradeoffs

- **Reservoir sampling** is uniform but *unweighted* and gives no recency bias — use weighted / time-decayed variants for recent-favouring samples.
- **Bloom filters** cannot delete (use a Counting Bloom filter) and must be sized for the expected `n`; overfilling collapses accuracy. No false negatives, but never zero false positives.
- **Flajolet-Martin** single estimators are high-variance; real systems use **HyperLogLog** (relative error ~ 1.04/√m). Good hashing is essential.
- **DGIM** (covered in the lesson) does windowed 1-counts in O(log²N) bits with ≤50% error.

## 7. Your turn

### Exercise 1 — Count-Min sketch (frequency estimation)

A Count-Min sketch estimates item frequencies with `d` rows of `w` counters. To **add** an item, increment one counter per row (row `i` at column `hash_i(x) % w`). To **estimate**, return the **minimum** across rows (collisions only ever inflate a count, so the min is the tightest).

In [ ]:
class CountMin:
    def __init__(self, d=5, w=200):
        self.d, self.w = d, w
        self.table = np.zeros((d, w), dtype=int)
    def _cols(self, x):
        for i in range(self.d):
            h = int(hashlib.md5((str(i) + ':' + str(x)).encode()).hexdigest(), 16)
            yield i, h % self.w
    def add(self, x):
        # TODO(you): increment table[i, col] for each (i, col) from self._cols(x)
        ...
    def estimate(self, x):
        # TODO(you): return the MINIMUM counter across the d rows for x
        return ...


In [ ]:
# Checks — run me
cm = CountMin(d=5, w=500)
truth = {}
for x in np.random.randint(0, 300, size=20000):
    cm.add(int(x)); truth[int(x)] = truth.get(int(x), 0) + 1
# Count-Min never underestimates, and with enough width the overestimate is small
assert all(cm.estimate(k) >= v for k, v in truth.items()), 'Count-Min must never underestimate'
err = np.mean([cm.estimate(k) - v for k, v in truth.items()])
assert err < 20, f'mean overestimate should be small, got {err:.1f}'
print('✅ Exercise 1 passed  (mean overestimate = %.2f)' % err)

<details>
<summary>💡 Show solution</summary>

```python
class CountMin:
    def __init__(self, d=5, w=200):
        self.d, self.w = d, w
        self.table = np.zeros((d, w), dtype=int)
    def _cols(self, x):
        for i in range(self.d):
            h = int(hashlib.md5((str(i) + ':' + str(x)).encode()).hexdigest(), 16)
            yield i, h % self.w
    def add(self, x):
        for i, col in self._cols(x):
            self.table[i, col] += 1
    def estimate(self, x):
        return min(self.table[i, col] for i, col in self._cols(x))
```

</details>

## Key takeaways

- **Reservoir sampling**: uniform `k`-sample in one pass (accept element `t` with prob `k/t`).
- **Bloom filter**: membership with no false negatives, FP rate `(1 - e^(-kn/m))^k`, optimal `k = (m/n)ln2`.
- **Flajolet-Martin / HyperLogLog**: distinct counts from max trailing zeros, `2^R`, in log space.
- **Count-Min**: frequency estimates that never underestimate.
- Next: [Online Learning & Regret](https://ml-viz-ruby.vercel.app/courses/streaming-ml/03-online-learning).